# YOLO26 Business Card Segmentation Training

This notebook prepares the current v1 dataset, validates segmentation labels, creates a train/validation split, writes a YOLO data file, and trains a segmentation model for business card extraction before OCR.

## 0. Colab Setup

In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9.0"])

print(f"Running in Colab: {IN_COLAB}")

## 1. Configuration

In [ ]:
from collections import Counter
from pathlib import Path
import math
import os
import random
import shutil
import sys

import cv2
import numpy as np
from PIL import Image
from IPython.display import Image as DisplayImage, display

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path("/content/drive/MyDrive/42174 AI Studio") if IN_COLAB else Path.cwd()
SOURCE_ROOT = PROJECT_ROOT / "Datasets" / "Business Card.yolo26_v1"
SOURCE_IMAGES = SOURCE_ROOT / "images"
SOURCE_LABELS = SOURCE_ROOT / "labels"

SPLIT_ROOT = PROJECT_ROOT / "Datasets" / "business_card_yolo26_segment_split"
PREVIEW_DIR = SPLIT_ROOT / "_preview"

CLASS_NAMES = ["Business-Card"]
VAL_RATIO = 0.20
SEED = 42
IMG_SIZE = 1024
BATCH_SIZE = 4
EPOCHS = 120
PATIENCE = 40

MODEL_WEIGHTS = PROJECT_ROOT / "yolo26n-seg.pt"
RUN_PROJECT = PROJECT_ROOT / "runs" / "segment"
RUN_NAME = "business_card_yolo26_segment_v1"

print(f"Project root: {PROJECT_ROOT}")
print(f"Source dataset: {SOURCE_ROOT}")
print(f"Split output: {SPLIT_ROOT}")

## 2. Dataset Utilities

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def image_files(image_dir: Path) -> list[Path]:
    return sorted(
        [path for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS],
        key=lambda path: path.name.lower(),
    )


def matching_label(image_path: Path, label_dir: Path) -> Path:
    return label_dir / f"{image_path.stem}.txt"


def parse_label_file(label_path: Path) -> list[list[float]]:
    objects = []
    for line_index, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
            raise ValueError(f"Invalid segmentation label at {label_path}:{line_index}")
        class_id = int(float(parts[0]))
        values = [float(value) for value in parts[1:]]
        if class_id != 0:
            raise ValueError(f"Unexpected class id {class_id} at {label_path}:{line_index}")
        if any(value < 0 or value > 1 or not math.isfinite(value) for value in values):
            raise ValueError(f"Out-of-range coordinate at {label_path}:{line_index}")
        objects.append([class_id, *values])
    if not objects:
        raise ValueError(f"Empty label file: {label_path}")
    return objects


def hardlink_or_copy(source: Path, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        target.unlink()
    try:
        os.link(source, target)
    except OSError:
        shutil.copy2(source, target)


def write_yolo_yaml(yaml_path: Path, dataset_root: Path, class_names: list[str]) -> None:
    lines = [
        f"path: {dataset_root.resolve().as_posix()}",
        "train: images/train",
        "val: images/val",
        "names:",
    ]
    lines.extend([f"  {index}: {name}" for index, name in enumerate(class_names)])
    yaml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def label_to_points(label_object: list[float], width: int, height: int) -> np.ndarray:
    values = label_object[1:]
    points = np.array(
        [[values[index] * width, values[index + 1] * height] for index in range(0, len(values), 2)],
        dtype=np.int32,
    )
    return points

## 3. Validate Source Dataset

In [ ]:
if not SOURCE_IMAGES.exists():
    raise FileNotFoundError(f"Missing image directory: {SOURCE_IMAGES}")
if not SOURCE_LABELS.exists():
    raise FileNotFoundError(f"Missing label directory: {SOURCE_LABELS}")

images = image_files(SOURCE_IMAGES)
if not images:
    raise FileNotFoundError(f"No images found in {SOURCE_IMAGES}")

missing_labels = [image_path.name for image_path in images if not matching_label(image_path, SOURCE_LABELS).exists()]
label_stems = {path.stem for path in SOURCE_LABELS.glob("*.txt")}
image_stems = {path.stem for path in images}
labels_without_images = sorted(label_stems - image_stems)

if missing_labels:
    raise ValueError(f"Images without labels: {missing_labels[:10]}")
if labels_without_images:
    raise ValueError(f"Labels without images: {labels_without_images[:10]}")

object_count = 0
point_counts = []
image_sizes = Counter()
orientation_counts = Counter()

for image_path in images:
    label_path = matching_label(image_path, SOURCE_LABELS)
    objects = parse_label_file(label_path)
    object_count += len(objects)
    point_counts.extend([(len(obj) - 1) // 2 for obj in objects])

    with Image.open(image_path) as image:
        image_sizes[f"{image.width}x{image.height}"] += 1
        orientation_counts[image.getexif().get(274, 1)] += 1

print(f"Images: {len(images)}")
print(f"Objects: {object_count}")
print(f"Point count range: {min(point_counts)} to {max(point_counts)}")
print(f"Image sizes: {dict(image_sizes)}")
print(f"EXIF orientations: {dict(orientation_counts)}")

if any(orientation != 1 for orientation in orientation_counts):
    print("Warning: non-default EXIF orientations were found. Check labels visually before training.")

## 4. Create Label Preview

In [ ]:
def draw_label_preview(image_path: Path, label_path: Path, tile_width: int = 420) -> np.ndarray:
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Unreadable image: {image_path}")
    height, width = image.shape[:2]
    for obj in parse_label_file(label_path):
        points = label_to_points(obj, width, height)
        cv2.polylines(image, [points.reshape((-1, 1, 2))], True, (0, 220, 0), max(2, round(max(width, height) / 500)))
        x1, y1 = points.min(axis=0)
        x2, y2 = points.max(axis=0)
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 130, 255), max(2, round(max(width, height) / 650)))

    cv2.putText(image, image_path.name[:50], (16, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(image, image_path.name[:50], (16, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    tile_height = max(1, round(height * tile_width / width))
    return cv2.resize(image, (tile_width, tile_height), interpolation=cv2.INTER_AREA)


def make_preview_sheet(sample_images: list[Path], output_path: Path, columns: int = 3) -> Path:
    tiles = [draw_label_preview(image_path, matching_label(image_path, SOURCE_LABELS)) for image_path in sample_images]
    tile_width = max(tile.shape[1] for tile in tiles)
    tile_height = max(tile.shape[0] for tile in tiles)
    rows = math.ceil(len(tiles) / columns)
    sheet = np.full((rows * tile_height, columns * tile_width, 3), 245, dtype=np.uint8)

    for index, tile in enumerate(tiles):
        row = index // columns
        column = index % columns
        y = row * tile_height
        x = column * tile_width
        sheet[y : y + tile.shape[0], x : x + tile.shape[1]] = tile

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), sheet)
    return output_path


rng = random.Random(SEED)
sample_images = images.copy()
rng.shuffle(sample_images)
preview_path = make_preview_sheet(sample_images[:24], PREVIEW_DIR / "source_labels_preview.jpg")
display(DisplayImage(filename=str(preview_path)))

## 5. Create Train/Validation Split

In [ ]:
RECREATE_SPLIT = True

if RECREATE_SPLIT:
    for child_name in ["images", "labels"]:
        child = SPLIT_ROOT / child_name
        if child.exists():
            shutil.rmtree(child)

for split in ["train", "val"]:
    (SPLIT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (SPLIT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

split_images = images.copy()
random.Random(SEED).shuffle(split_images)
val_count = max(1, round(len(split_images) * VAL_RATIO))
val_set = {path.name for path in split_images[:val_count]}

split_counts = Counter()
for image_path in images:
    split = "val" if image_path.name in val_set else "train"
    label_path = matching_label(image_path, SOURCE_LABELS)
    hardlink_or_copy(image_path, SPLIT_ROOT / "images" / split / image_path.name)
    hardlink_or_copy(label_path, SPLIT_ROOT / "labels" / split / label_path.name)
    split_counts[split] += 1

write_yolo_yaml(SPLIT_ROOT / "data.yaml", SPLIT_ROOT, CLASS_NAMES)

print(f"Train images: {split_counts['train']}")
print(f"Validation images: {split_counts['val']}")
print(f"Data YAML: {SPLIT_ROOT / 'data.yaml'}")

## 6. Validate Split Output

In [ ]:
for split in ["train", "val"]:
    split_image_dir = SPLIT_ROOT / "images" / split
    split_label_dir = SPLIT_ROOT / "labels" / split
    split_image_files = image_files(split_image_dir)
    split_label_files = sorted(split_label_dir.glob("*.txt"), key=lambda path: path.name.lower())

    image_stems = {path.stem for path in split_image_files}
    label_stems = {path.stem for path in split_label_files}
    if image_stems != label_stems:
        raise ValueError(f"Image/label mismatch in {split}")

    for label_path in split_label_files:
        parse_label_file(label_path)

    print(f"{split}: {len(split_image_files)} images, {len(split_label_files)} labels")

print((SPLIT_ROOT / "data.yaml").read_text(encoding="utf-8"))

## 7. Environment Check

In [ ]:
import importlib.util

if importlib.util.find_spec("ultralytics") is None:
    raise ModuleNotFoundError("Install ultralytics before training: pip install 'ultralytics>=8.3,<9.0'")

import torch
from ultralytics import YOLO

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 8. Train YOLO26 Segmentation Model

In [ ]:
from ultralytics import YOLO

data_yaml = SPLIT_ROOT / "data.yaml"
if not data_yaml.exists():
    raise FileNotFoundError(f"Run the split creation cell first: {data_yaml}")

weights = str(MODEL_WEIGHTS) if MODEL_WEIGHTS.exists() else "yolo26n-seg.pt"
print(f"Model weights: {weights}")

model = YOLO(weights)
results = model.train(
    task="segment",
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    workers=0,
    patience=PATIENCE,
    project=str(RUN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    cos_lr=True,
    close_mosaic=15,
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.35,
    degrees=12.0,
    translate=0.10,
    scale=0.45,
    shear=2.0,
    perspective=0.0008,
    fliplr=0.30,
    flipud=0.0,
    mosaic=0.40,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    overlap_mask=True,
    mask_ratio=4,
    plots=True,
)

print(f"Training run: {RUN_PROJECT / RUN_NAME}")

## 9. Validate Best Weights

In [ ]:
from ultralytics import YOLO

best_weights = RUN_PROJECT / RUN_NAME / "weights" / "best.pt"
if not best_weights.exists():
    raise FileNotFoundError(f"Best weights not found: {best_weights}")

model = YOLO(str(best_weights))
metrics = model.val(
    data=str(SPLIT_ROOT / "data.yaml"),
    imgsz=IMG_SIZE,
    split="val",
    project=str(RUN_PROJECT),
    name=f"{RUN_NAME}_val",
    exist_ok=True,
    plots=True,
)
metrics

## 10. Prediction Preview

In [ ]:
from ultralytics import YOLO

best_weights = RUN_PROJECT / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best_weights))
preview_source = SPLIT_ROOT / "images" / "val"

predictions = model.predict(
    source=str(preview_source),
    imgsz=IMG_SIZE,
    conf=0.25,
    retina_masks=True,
    save=True,
    project=str(RUN_PROJECT),
    name=f"{RUN_NAME}_preview",
    exist_ok=True,
)

print(f"Prediction preview: {RUN_PROJECT / (RUN_NAME + '_preview')}")